# XPLT Explorer — Catheter/Urethra Simulation

Extracts mesh data and contact pressure from `sample.xplt` + `sample.feb`.

In [1]:
import struct
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path

FEB_FILE = Path('sample.feb')
XPLT_FILE = Path('sample.xplt')

## 1. Parse .feb — Material and Domain Mapping

In [2]:
tree = ET.parse(FEB_FILE)
root = tree.getroot()

# Material id -> name
mat_id_to_name = {}
mat_name_to_id = {}
for mat in root.findall('./Material/material'):
    mid = int(mat.get('id'))
    name = mat.get('name')
    mat_id_to_name[mid] = name
    mat_name_to_id[name] = mid

print('Materials:')
for mid, name in sorted(mat_id_to_name.items()):
    print(f'  id={mid}  name="{name}"')

Materials:
  id=1  name="catheter_body"
  id=2  name="catheter_tip"
  id=3  name="zone2"
  id=4  name="zone3"
  id=5  name="zone4"
  id=6  name="zone1"


In [3]:
# Part name -> material name (from MeshDomains)
part_to_mat = {}
for domain in root.findall('./MeshDomains/SolidDomain'):
    part_to_mat[domain.get('name')] = domain.get('mat')

print('Parts -> Materials:')
for part, mat in part_to_mat.items():
    print(f'  {part} -> {mat}')

Parts -> Materials:
  Part18 -> zone1
  Part26 -> zone2
  Part29 -> zone4
  Part30 -> zone3
  Part31 -> catheter_tip
  Part32 -> catheter_body


In [4]:
# catheter_slidePrimary surface facets from .feb
# Node IDs here are 1-indexed (global FEB IDs)
feb_surface_facets = {}  # surface_name -> list of (facet_id, [node_ids])
surfaces_of_interest = {'catheter_slidePrimary', 'catheter_slideSecondary'}

for surf in root.findall('./Mesh/Surface'):
    name = surf.get('name')
    if name in surfaces_of_interest:
        facets = []
        for tri in surf.findall('tri3'):
            fid = int(tri.get('id'))
            nodes = list(map(int, tri.text.split(',')))
            facets.append((fid, nodes))
        feb_surface_facets[name] = facets
        print(f'Surface "{name}": {len(facets)} facets')

Surface "catheter_slidePrimary": 23734 facets
Surface "catheter_slideSecondary": 10571 facets


## 2. Parse .xplt Binary — Custom Reader

FEBio xplt v53 format: blocks of `[tag:4B, size:4B, data:sizeB]`.
Top-level: ROOT header, MESH, then one STATE block per timestep.

In [6]:
# --- Low-level block parser ---

def iter_blocks(data: bytes):
    """Yield (tag, size, chunk) for each block in a byte buffer."""
    pos = 0
    while pos + 8 <= len(data):
        tag  = struct.unpack_from('<I', data, pos)[0]
        size = struct.unpack_from('<I', data, pos + 4)[0]
        pos += 8
        if pos + size > len(data):
            break
        yield tag, size, data[pos:pos + size]
        pos += size

def find_block(data: bytes, target_tag: int):
    """Return the first block chunk matching target_tag, or None."""
    for tag, size, chunk in iter_blocks(data):
        if tag == target_tag:
            return chunk
    return None

# xplt tag constants
TAG = dict(
    ROOT           = 0x01000000,
    MESH           = 0x01040000,
    NODE_SECTION   = 0x01041000,
    NODE_HEADER    = 0x01041100,
    NODE_N_NODES   = 0x01041101,
    NODE_COORDS    = 0x01041200,
    DOMAIN_SECTION = 0x01042000,
    DOMAIN         = 0x01042100,
    DOMAIN_HEADER  = 0x01042101,
    DOM_ELEM_TYPE  = 0x01042102,
    DOM_PART_ID    = 0x01042103,
    DOM_N_ELEMS    = 0x01032104,
    DOM_NAME       = 0x01032105,
    DOM_ELEM_LIST  = 0x01042200,
    ELEMENT        = 0x01042201,
    SURF_SECTION   = 0x01043000,
    SURFACE        = 0x01043100,
    SURF_HEADER    = 0x01043101,
    SURF_ID        = 0x01043102,
    SURF_N_FACETS  = 0x01043103,
    SURF_NAME      = 0x01043104,
    FACET_LIST     = 0x01043200,
    FACET          = 0x01043201,
    STATE          = 0x02000000,
    STATE_HEADER   = 0x02010000,
    STATE_TIME     = 0x02010002,
    STATE_DATA     = 0x02020000,
    STATE_VARIABLE = 0x02020001,
    STATE_VAR_ID   = 0x02020002,
    STATE_VAR_DATA = 0x02020003,
    NODE_DATA      = 0x02020300,
    ELEMENT_DATA   = 0x02020400,
    SURFACE_DATA   = 0x02020500,
)

print('Tag constants loaded.')

Tag constants loaded.


In [7]:
# --- Read raw bytes once ---
with open(XPLT_FILE, 'rb') as f:
    raw = f.read()

print(f'File size: {len(raw)/1e6:.1f} MB')

# Locate top-level blocks (skip 4-byte magic "BEF\0")
top_blocks = {}
state_positions = []  # list of (pos, size) for each STATE block
pos = 4
while pos + 8 < len(raw):
    tag  = struct.unpack_from('<I', raw, pos)[0]
    size = struct.unpack_from('<I', raw, pos + 4)[0]
    if tag == TAG['STATE']:
        state_positions.append((pos, size))
    else:
        top_blocks[tag] = (pos, size)
    pos += 8 + size

print(f'Top-level blocks: {[hex(k) for k in top_blocks]}')
print(f'State (timestep) blocks: {len(state_positions)}')

mesh_pos, mesh_size = top_blocks[TAG['MESH']]
mesh_data = raw[mesh_pos + 8 : mesh_pos + 8 + mesh_size]

File size: 2265.5 MB
Top-level blocks: ['0x1000000', '0x1040000']
State (timestep) blocks: 180


## 3. Extract Node Coordinates

In [8]:
# Node section: each node stored as [node_id (int as float32), x, y, z]
node_section = find_block(mesh_data, TAG['NODE_SECTION'])
node_coords_raw = find_block(node_section, TAG['NODE_COORDS'])

node_arr = np.frombuffer(node_coords_raw, dtype='<f4').reshape(-1, 4)
# node_arr[:, 0] = node global ID stored as int32 bits (subnormal float)
# node_arr[:, 1:4] = x, y, z coordinates
node_ids_global = np.frombuffer(node_arr[:, 0].tobytes(), dtype='<u4')  # 0-indexed position -> global FEB node ID
coords = node_arr[:, 1:4].astype(np.float64)  # shape (N, 3)

print(f'Nodes: {len(coords)}')
print(f'Coord range X: [{coords[:,0].min():.2f}, {coords[:,0].max():.2f}]')
print(f'Coord range Y: [{coords[:,1].min():.2f}, {coords[:,1].max():.2f}]')
print(f'Coord range Z: [{coords[:,2].min():.2f}, {coords[:,2].max():.2f}]')
print(f'Global node ID range: {node_ids_global.min()} – {node_ids_global.max()}')

Nodes: 52245
Coord range X: [-68.83, 71.27]
Coord range Y: [-15.11, 139.83]
Coord range Z: [-277.02, 402.60]
Global node ID range: 74470 – 126714


## 4. Extract Element Domains (Parts)

In [9]:
# XPLT part_id corresponds to material_id in .feb Material section
domain_section = find_block(mesh_data, TAG['DOMAIN_SECTION'])
domains = []  # list of dicts

for tag, size, chunk in iter_blocks(domain_section):
    if tag != TAG['DOMAIN']:
        continue
    dom = {}
    for t2, s2, c2 in iter_blocks(chunk):
        if t2 == TAG['DOMAIN_HEADER']:
            for t3, s3, c3 in iter_blocks(c2):
                if   t3 == TAG['DOM_ELEM_TYPE']: dom['elem_type'] = struct.unpack_from('<I', c3)[0]
                elif t3 == TAG['DOM_PART_ID']:   dom['part_id']   = struct.unpack_from('<I', c3)[0]
                elif t3 == TAG['DOM_N_ELEMS']:   dom['n_elems']   = struct.unpack_from('<I', c3)[0]
                elif t3 == TAG['DOM_NAME']:      dom['name']      = c3.rstrip(b'\x00').decode('latin-1').strip()
        elif t2 == TAG['DOM_ELEM_LIST']:
            # Each ELEMENT: [elem_id (int), n0, n1, n2, n3] for tet4 = 5×4 bytes
            elems = []
            for t3, s3, c3 in iter_blocks(c2):
                if t3 == TAG['ELEMENT']:
                    n = s3 // 4
                    elems.append(struct.unpack_from(f'<{n}I', c3))
            dom['elements'] = np.array(elems, dtype=np.int32)  # shape (n_elems, 5) for tet4
    dom['mat_name'] = mat_id_to_name.get(dom.get('part_id'), 'unknown')
    domains.append(dom)

print(f'Domains found: {len(domains)}')
for d in domains:
    print(f'  part_id={d["part_id"]}  mat="{d["mat_name"]}"  n_elems={d["n_elems"]}  xplt_name="{d["name"]}"')

Domains found: 6
  part_id=6  mat="zone1"  n_elems=29018  xplt_name="   Part18"
  part_id=3  mat="zone2"  n_elems=48415  xplt_name="   Part26"
  part_id=5  mat="zone4"  n_elems=10126  xplt_name="   Part29"
  part_id=4  mat="zone3"  n_elems=23520  xplt_name="   Part30"
  part_id=2  mat="catheter_tip"  n_elems=9292  xplt_name="   Part31"
  part_id=1  mat="catheter_body"  n_elems=77787  xplt_name="   Part32"


## 5. Extract Catheter Mesh (catheter_body + catheter_tip)

In [11]:
CATHETER_MATS = {'catheter_body', 'catheter_tip'}

cath_domains = [d for d in domains if d['mat_name'] in CATHETER_MATS]

for d in cath_domains:
    elems = d['elements']  # shape (n, 5): [elem_id, n0, n1, n2, n3]
    # node indices are 0-based positions in coords array
    node_indices = np.unique(elems[:, 1:])  # all unique node positions used
    cath_coords = coords[node_indices]
    print(f'Material: {d["mat_name"]}')
    print(f'  Elements: {len(elems)}')
    print(f'  Unique nodes: {len(node_indices)}')
    print(f'  X range: [{cath_coords[:,0].min():.2f}, {cath_coords[:,0].max():.2f}]')
    print(f'  Y range: [{cath_coords[:,1].min():.2f}, {cath_coords[:,1].max():.2f}]')
    print(f'  Z range: [{cath_coords[:,2].min():.2f}, {cath_coords[:,2].max():.2f}]')
    print()

Material: catheter_tip
  Elements: 9292
  Unique nodes: 2535
  X range: [-1.90, 2.70]
  Y range: [-2.61, 1.99]
  Z range: [-0.40, 21.10]

Material: catheter_body
  Elements: 77787
  Unique nodes: 25496
  X range: [-1.90, 2.70]
  Y range: [-2.61, 1.99]
  Z range: [18.10, 402.60]



In [ ]:
# Combined catheter node indices and coordinates
all_cath_node_indices = np.unique(
    np.concatenate([d['elements'][:, 1:].ravel() for d in cath_domains])
)
cath_coords_all = coords[all_cath_node_indices]

print(f'Total catheter nodes (body+tip): {len(all_cath_node_indices)}')
print(f'Total catheter elements (body+tip): {sum(len(d["elements"]) for d in cath_domains)}')

## 6. Extract Surface Facets

In [12]:
surf_section = find_block(mesh_data, TAG['SURF_SECTION'])
xplt_surfaces = {}  # surface_id -> dict with name, n_facets, facets array

for tag, size, chunk in iter_blocks(surf_section):
    if tag != TAG['SURFACE']:
        continue
    surf = {}
    for t2, s2, c2 in iter_blocks(chunk):
        if t2 == TAG['SURF_HEADER']:
            for t3, s3, c3 in iter_blocks(c2):
                if   t3 == TAG['SURF_ID']:       surf['id']       = struct.unpack_from('<I', c3)[0]
                elif t3 == TAG['SURF_N_FACETS']: surf['n_facets'] = struct.unpack_from('<I', c3)[0]
                elif t3 == TAG['SURF_NAME']:     surf['name']     = c3.rstrip(b'\x00').decode('latin-1').strip()
        elif t2 == TAG['FACET_LIST']:
            facets = []
            for t3, s3, c3 in iter_blocks(c2):
                if t3 == TAG['FACET']:
                    # Each tri3 facet: [facet_id, n_nodes=3, n0, n1, n2] (5 × uint32)
                    vals = struct.unpack_from('<5I', c3)
                    facets.append(vals[2:5])  # keep only node indices
            surf['facets'] = np.array(facets, dtype=np.int32)  # shape (n, 3)
    xplt_surfaces[surf['id']] = surf

print(f'Surfaces parsed: {len(xplt_surfaces)}')
for sid, s in sorted(xplt_surfaces.items()):
    print(f'  id={sid}  name="{s["name"]}"  n_facets={s["n_facets"]}')

Surfaces parsed: 13
  id=1  name="   cath_tail_zero_rot"  n_facets=38
  id=2  name="   cath_tail_xy_zero_disp"  n_facets=38
  id=3  name="   catheter_slidePrimary"  n_facets=23734
  id=4  name="   catheter_slideSecondary"  n_facets=10571
  id=5  name="!   cath_body_tip_tied_contactPrimary"  n_facets=27470
  id=6  name="#   cath_body_tip_tied_contactSecondary"  n_facets=2318
  id=7  name="   z1z2TiedPrimary"  n_facets=517
  id=8  name="   z1z2TiedSecondary"  n_facets=517
  id=9  name="   z2z3TiedPrimary"  n_facets=1233
  id=10  name="   z3z4TiedSecondary"  n_facets=563
  id=11  name="   zero_disp_bladder"  n_facets=536
  id=12  name="   z2z3TiedSecondary"  n_facets=1233
  id=13  name="   z3z4TiedPrimary"  n_facets=563


In [ ]:
# catheter_slidePrimary = surface id 3
cath_slide_primary = xplt_surfaces[3]
facets = cath_slide_primary['facets']  # shape (23734, 3) — 0-indexed into coords

print(f'catheter_slidePrimary facets: {len(facets)}')
print(f'First 5 facets (node indices): {facets[:5].tolist()}')

# Compute facet centroids
f_coords = coords[facets]  # shape (N, 3, 3)
centroids = f_coords.mean(axis=1)  # shape (N, 3)

print(f'Centroid Z range: [{centroids[:,2].min():.2f}, {centroids[:,2].max():.2f}] mm')
print(f'Centroid Z tip region (< 50mm): {(centroids[:,2] < 50).sum()} facets')

catheter_slidePrimary facets: 23734
First 5 facets (node indices): [[24255, 24644, 24645], [24678, 24680, 24679], [24664, 24660, 24650], [25915, 25936, 25916], [25916, 24343, 24342]]
Centroid Z range: [-0.28, 402.41] mm
Centroid Z tip region (< -50mm): 0 facets


## 7. Parse All States and Extract Contact Pressure

**Binary layout** (xplt v53 surface data):
Each contact pair appears as two surface blocks (primary then secondary):
`[surface_id (int as f32), byte_count (int as f32), f0, f1, ..., f(n-1)]`

Contact order in the file: catheter_slide, cath_body_tip_tied, z1z2, z2z3, z3z4

`catheter_slidePrimary` data starts at **offset 2** in the flat SURFACE_DATA array.

In [ ]:
# Build a map of surface_id -> offset in the flat surface_data array
# Structure: each surface stored as [int_id_as_f32, byte_count_as_f32, f0..f(n-1)]
# Contact pair order in xplt (determined empirically from facet_area analysis):
#   catheter_slide:         primary=id3(23734), secondary=id4(10571)
#   cath_body_tip_tied:     primary=id5(27470), secondary=id6(2318)
#   z1z2Tied:               primary=id7(517),  secondary=id8(517)
#   z2z3Tied:               primary=id9(1233), secondary=id12(1233)
#   z3z4Tied:               primary=id13(563), secondary=id10(563)

CONTACT_ORDER = [
    (3, 4),    # catheter_slide
    (5, 6),    # cath_body_tip_tied_contact
    (7, 8),    # z1z2Tied
    (9, 12),   # z2z3Tied
    (13, 10),  # z3z4Tied
]

surface_data_offset = {}  # surface_id -> start offset in flat array (after 2-float header)
pos = 0
for primary_id, secondary_id in CONTACT_ORDER:
    for sid in (primary_id, secondary_id):
        n = xplt_surfaces[sid]['n_facets']
        pos += 2  # skip [surface_id, byte_count] header
        surface_data_offset[sid] = pos
        pos += n

print('Surface data offsets in flat array:')
for sid, offset in surface_data_offset.items():
    n = xplt_surfaces[sid]['n_facets']
    print(f'  surface {sid} "{xplt_surfaces[sid]["name"]}": offset={offset}, n={n}')

In [ ]:
# DIC_SURFACE variable IDs (from xplt dictionary, confirmed by order in ROOT block):
VAR_CONTACT_PRESSURE = 1  # scalar per facet

def extract_surface_variable(state_chunk: bytes, var_id: int) -> np.ndarray:
    """Extract flat surface data array for a given variable ID from a STATE block."""
    state_data_chunk = find_block(state_chunk, TAG['STATE_DATA'])
    if state_data_chunk is None:
        return None
    surf_data_chunk = find_block(state_data_chunk, TAG['SURFACE_DATA'])
    if surf_data_chunk is None:
        return None
    for tag, size, chunk in iter_blocks(surf_data_chunk):
        if tag != TAG['STATE_VARIABLE']:
            continue
        vid = None
        arr = None
        for t2, s2, c2 in iter_blocks(chunk):
            if   t2 == TAG['STATE_VAR_ID']:   vid = struct.unpack_from('<I', c2)[0]
            elif t2 == TAG['STATE_VAR_DATA']: arr = np.frombuffer(c2, dtype='<f4')
        if vid == var_id:
            return arr
    return None

def extract_timestep(state_chunk: bytes) -> float:
    header = find_block(state_chunk, TAG['STATE_HEADER'])
    time_bytes = find_block(header, TAG['STATE_TIME'])
    return float(struct.unpack_from('<f', time_bytes)[0])

print('Helper functions defined.')

In [ ]:
# Extract contact pressure for catheter_slidePrimary at ALL timesteps
# This may take ~30-60 seconds for 180 states

N_FACETS = xplt_surfaces[3]['n_facets']  # 23734
CP_START = surface_data_offset[3]         # 2
CP_END   = CP_START + N_FACETS            # 23736

timesteps = []
contact_pressure = []  # list of np.array shape (N_FACETS,)

for i, (spos, ssize) in enumerate(state_positions):
    state_chunk = raw[spos + 8 : spos + 8 + ssize]
    t = extract_timestep(state_chunk)
    flat = extract_surface_variable(state_chunk, VAR_CONTACT_PRESSURE)
    if flat is not None:
        cp = flat[CP_START:CP_END].copy()
    else:
        cp = np.zeros(N_FACETS, dtype=np.float32)
    timesteps.append(t)
    contact_pressure.append(cp)
    if i % 30 == 0:
        print(f'  Parsed state {i+1}/{len(state_positions)}, t={t:.3f}')

timesteps = np.array(timesteps)
cp_matrix = np.stack(contact_pressure)  # shape (n_timesteps, n_facets)

print(f'\nDone. Shape: {cp_matrix.shape}  (timesteps × facets)')
print(f'Time range: {timesteps[0]:.3f} → {timesteps[-1]:.3f}')
print(f'Max contact pressure overall: {cp_matrix.max():.6f} MPa')

## 8. Explore Results

In [ ]:
# At the final timestep: which facets have non-zero contact pressure?
final_cp = cp_matrix[-1]  # shape (N_FACETS,)
nz_mask = final_cp > 0

print(f'Final timestep t={timesteps[-1]:.3f}:')
print(f'  Facets with contact pressure > 0: {nz_mask.sum()} / {N_FACETS}')
print(f'  Max pressure: {final_cp.max():.6f} MPa')
print(f'  Mean (non-zero): {final_cp[nz_mask].mean():.6f} MPa')

In [ ]:
# Build a DataFrame: one row per facet, with centroid coords + contact pressure at each timestep
import pandas as pd

df_facets = pd.DataFrame({
    'facet_idx': np.arange(N_FACETS),
    'cx': centroids[:, 0],
    'cy': centroids[:, 1],
    'cz': centroids[:, 2],
    'cp_final': final_cp,
})

print(df_facets.describe())
print()
print('Top 10 facets by contact pressure at final timestep:')
print(df_facets.nlargest(10, 'cp_final')[['facet_idx', 'cx', 'cy', 'cz', 'cp_final']])

In [ ]:
# Filter to catheter_tip zone: facets whose centroids overlap catheter_tip Z range
# Find Z range of catheter_tip
tip_domain = next(d for d in domains if d['mat_name'] == 'catheter_tip')
tip_node_idx = np.unique(tip_domain['elements'][:, 1:].ravel())
tip_z_min = coords[tip_node_idx, 2].min()
tip_z_max = coords[tip_node_idx, 2].max()

print(f'catheter_tip Z range: [{tip_z_min:.2f}, {tip_z_max:.2f}] mm')

tip_facet_mask = (centroids[:, 2] >= tip_z_min) & (centroids[:, 2] <= tip_z_max)
print(f'Facets in catheter_tip Z range: {tip_facet_mask.sum()}')

df_tip = df_facets[tip_facet_mask].copy()
print(f'\nTip facets with cp > 0 at final step: {(df_tip.cp_final > 0).sum()}')
print(f'Max cp in tip zone: {df_tip.cp_final.max():.6f} MPa')

In [ ]:
# Contact pressure over time — mean and max across all catheter_slidePrimary facets
cp_mean = cp_matrix.mean(axis=1)          # (n_timesteps,)
cp_max  = cp_matrix.max(axis=1)           # (n_timesteps,)
cp_tip_max = cp_matrix[:, tip_facet_mask].max(axis=1)  # max in tip zone

df_time = pd.DataFrame({
    'time': timesteps,
    'cp_mean_all': cp_mean,
    'cp_max_all': cp_max,
    'cp_max_tip': cp_tip_max,
})

print(df_time.tail(10))

In [ ]:
# Optional: plot if matplotlib is available
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: contact pressure vs time
    ax = axes[0]
    ax.plot(df_time.time, df_time.cp_max_all,  label='Max (all)')
    ax.plot(df_time.time, df_time.cp_max_tip,  label='Max (tip zone)', linestyle='--')
    ax.plot(df_time.time, df_time.cp_mean_all, label='Mean (all)', linestyle=':')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Contact Pressure (MPa)')
    ax.set_title('Contact Pressure vs Time')
    ax.legend()
    ax.grid(True)

    # Right: scatter of facet centroids coloured by final contact pressure
    ax2 = axes[1]
    sc = ax2.scatter(df_facets.cz, df_facets.cy, c=df_facets.cp_final,
                     cmap='hot', s=1, vmin=0, vmax=df_facets.cp_final.quantile(0.99))
    plt.colorbar(sc, ax=ax2, label='Contact Pressure (MPa)')
    ax2.set_xlabel('Z (mm)')
    ax2.set_ylabel('Y (mm)')
    ax2.set_title('Facet Contact Pressure at Final Timestep')

    plt.tight_layout()
    plt.savefig('contact_pressure_overview.png', dpi=150)
    plt.show()
    print('Plot saved to contact_pressure_overview.png')
except ImportError:
    print('matplotlib not installed — skipping plot.')

In [ ]:
# Export full contact pressure matrix to CSV if needed
# Rows = timesteps, Columns = facets
# Uncomment to export (large file):
# pd.DataFrame(cp_matrix, index=timesteps).to_csv('cp_matrix.csv')

# Export final timestep facet data
df_facets.to_csv('catheter_slide_primary_final.csv', index=False)
print('Saved: catheter_slide_primary_final.csv')